# Error analysis
Run test evaluation first. This notebook ranks actual patient results and provides a framework for reviewing false positives, false negatives, small-lesion misses, boundary errors, and fragmented predictions without making clinical claims.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data.nifti import load_nifti
from src.data.preprocessing import convert_brats_mask_to_binary
from src.visualization.plotting import overlay_mask
results = pd.read_csv(ROOT / 'outputs/evaluation/per_patient.csv')
worst = results.nsmallest(10, 'dice')
display(worst[['patient_id','dice','iou','precision','recall','specificity']])

In [ ]:
metadata = pd.read_csv(ROOT / 'data/metadata/dataset_metadata.csv').set_index('patient_id')
patient_id = worst.iloc[0].patient_id
row = metadata.loc[patient_id]
flair = load_nifti(row.flair_path).array
ground_truth = convert_brats_mask_to_binary(load_nifti(row.seg_path).array)
prediction = load_nifti(ROOT / 'outputs/predictions' / f'{patient_id}_pred_seg.nii.gz').array > 0
z = int(np.argmax(np.logical_xor(prediction, ground_truth).sum(axis=(0,1))))
panels = [('FLAIR', flair[:,:,z], 'gray'), ('Ground truth', ground_truth[:,:,z], 'gray'), ('Prediction', prediction[:,:,z], 'gray'), ('GT overlay', overlay_mask(flair[:,:,z], ground_truth[:,:,z], (0.1,1,0.1)), None), ('Prediction overlay', overlay_mask(flair[:,:,z], prediction[:,:,z]), None)]
fig, axes = plt.subplots(1, 5, figsize=(18,4))
for axis, (title, image, cmap) in zip(axes, panels): axis.imshow(image, cmap=cmap); axis.set_title(title); axis.axis('off')
fig.suptitle(f'{patient_id} — largest disagreement slice {z}'); plt.tight_layout()

## Review checklist
For each low-Dice patient, inspect FLAIR, ground truth, prediction, and both overlays on the largest-error slices. Assign observable error types only: false positive, false negative, small-lesion miss, boundary error, or fragmented prediction. Do not infer clinical causes from these images.